In [2]:
import CoolProp.CoolProp as CP

def print_separator(char='-', length=95):
    print(char * length)

def calculate_and_print_table():
    # Standard Atmosphärendruck in Pascal
    P_ATM = 101325 

    # Daten aus dem Paper (Table 1)
    # Format: (Test Nr, T_air [C], RH_air [%], T_surface [C], Paper_Supercooling_Val [C])
    data = [
        (1, 7.0, 85.0, -10.0, 14.5),
        (2, 2.5, 85.0, -10.0, 10.0),
        (3, 2.5, 74.0,  -5.0,  5.0)
    ]

    print("\n" + "="*35 + " NACHRECHNUNG DER PAPER-DATEN " + "="*30)
    print(f"{'Test':<5} {'T_air':<8} {'RH':<6} {'T_surf':<8} | {'T_tau (Calc)':<12} {'Supercool (Calc)':<18} {'Paper Wert':<12} {'Status'}")
    print_separator()

    for test_id, t_air, rh, t_surf, paper_val in data:
        # CoolProp benötigt SI-Einheiten (Kelvin, 0-1 für RH)
        # HAPropsSI Inputs: "Tdp" (Taupunkt) gesucht, gegeben "T", "R" (RH), "P"
        t_dew_k = CP.HAPropsSI('Tdp', 'T', t_air + 273.15, 'R', rh / 100.0, 'P', P_ATM)
        t_dew_c = t_dew_k - 273.15

        # Berechnung des "Supercooling Degree" (T_tau - T_surf)
        supercooling_calc = t_dew_c - t_surf
        
        # Abweichung prüfen
        diff = abs(supercooling_calc - paper_val)
        status = "OK" if diff < 0.3 else "ABWEICHUNG!"

        print(f"{test_id:<5} {t_air:<8.1f} {rh:<6.0f} {t_surf:<8.1f} | {t_dew_c:<12.2f} {supercooling_calc:<18.2f} {paper_val:<12.1f} {status}")

    print_separator()
    print("\n")

def solve_mystery_test_3():
    P_ATM = 101325
    
    # Zielwerte aus dem Paper für Test 3
    target_supercooling = 5.0
    t_surf = -5.0
    t_air = 2.5
    
    # Wenn Supercooling = 5.0 und T_surf = -5.0, muss der Taupunkt 0.0 C sein.
    target_t_dew = t_surf + target_supercooling # ergibt 0.0 C
    
    print("="*35 + " ANALYSE TEST NR. 3 " + "="*36)
    print(f"Ziel-Supercooling laut Paper: {target_supercooling:.1f} K")
    print(f"Dafür nötiger Taupunkt:       {target_t_dew:.1f} °C")
    print("-" * 95)
    
    # Rückwärtsrechnung: Welche RH ist nötig, um bei T_air=2.5 C einen Taupunkt von 0 C zu haben?
    # HAPropsSI Inputs: "R" (RH) gesucht, gegeben "T", "Tdp", "P"
    required_rh = CP.HAPropsSI('R', 'T', t_air + 273.15, 'Tdp', target_t_dew + 273.15, 'P', P_ATM)
    required_rh_percent = required_rh * 100

    print(f"Gegebene T_Luft:              {t_air:.1f} °C")
    print(f"Paper Angabe RH:              74.0 %  -> Führt zu Supercooling von 3.3 K (Falsch)")
    print(f"BERECHNETE NÖTIGE RH:         {required_rh_percent:.2f} %  -> Führt zu Supercooling von 5.0 K (Korrekt)")
    print("="*95 + "\n")

if __name__ == "__main__":
    calculate_and_print_table()
    solve_mystery_test_3()


=================================== NACHRECHNUNG DER PAPER-DATEN ==============================
Test  T_air    RH     T_surf   | T_tau (Calc) Supercool (Calc)   Paper Wert   Status
-----------------------------------------------------------------------------------------------
1     7.0      85     -10.0    | 4.65         14.65              14.5         OK
2     2.5      85     -10.0    | 0.24         10.24              10.0         OK
3     2.5      74     -5.0     | -1.47        3.53               5.0          ABWEICHUNG!
-----------------------------------------------------------------------------------------------


=================================== ANALYSE TEST NR. 3 ====================================
Ziel-Supercooling laut Paper: 5.0 K
Dafür nötiger Taupunkt:       0.0 °C
-----------------------------------------------------------------------------------------------
Gegebene T_Luft:              2.5 °C
Paper Angabe RH:              74.0 %  -> Führt zu Supercooling von 3.3 K (